# ClaimCheck — QLoRA Fine-Tuning Notebook
**Qwen2.5-3B-Instruct · Rank-8 LoRA · Free T4 GPU**

This notebook fine-tunes Qwen2.5-3B-Instruct with QLoRA (4-bit quantization + LoRA adapters)
for structured field extraction from Indian hospital claim forms and discharge summaries.

**What the model learns:** Given a document (claim form or discharge summary), extract fields
(patient name, diagnosis, procedure, hospital, dates, amounts) with exact quotes from the text.

**Run on:** Kaggle (T4 GPU) or Google Colab (T4 GPU, free tier).

> Woxsen University · SLM Assignment · ClaimCheck Project

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────
# Run this cell first. Restart runtime if prompted.
# transformers is capped below 5.0 to avoid numpy/scipy conflicts.
# Everything goes in ONE pip install so peft/trl can't pull in transformers 5.x.

!pip install -q \
    "transformers>=4.46.0,<5.0.0" \
    "peft>=0.13.0,<1.0.0" \
    "trl>=0.12.0,<1.0.0" \
    "bitsandbytes>=0.44.0" \
    "datasets>=3.0.0,<5.0.0" \
    "accelerate>=1.0.0" \
    sentencepiece \
    protobuf

import transformers, trl, peft, torch, bitsandbytes
print(f"✅ All dependencies installed")
print(f"   transformers={transformers.__version__}")
print(f"   trl={trl.__version__}")
print(f"   peft={peft.__version__}")
print(f"   torch={torch.__version__}")
print(f"   bitsandbytes={bitsandbytes.__version__}")
assert int(transformers.__version__.split('.')[0]) < 5, \
    f"❌ transformers {transformers.__version__} is too new — needs <5.0.0. Restart kernel and re-run."
print("✅ transformers version OK (< 5.0)")
assert torch.cuda.is_available(), "❌ No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Step 2: Configuration ─────────────────────────────────────────────────
# All hyperparameters in one place. Adjust if needed.

CONFIG = {
    # Model
    "model_id": "Qwen/Qwen2.5-3B-Instruct",
    "max_seq_length": 2048,

    # QLoRA
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],

    # Training
    "num_epochs": 1,                    # 1 epoch ≈ 188 steps ≈ 2.5 hrs on free T4
    "batch_size": 2,
    "gradient_accumulation_steps": 8,   # effective batch = 2 * 8 = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "lr_scheduler": "cosine",
    "max_grad_norm": 1.0,
    "fp16": False,                      # DISABLED — Qwen2.5's 4-bit layers produce bf16 at runtime, crashes GradScaler
    "bf16": False,                      # T4 doesn't support bf16

    # Data
    "train_size": 1500,
    "val_size": 150,
    "test_size": 400,
    "mismatch_rate": 0.30,

    # Output
    "output_dir": "./claimcheck-qwen2.5-3b-lora",
    "logging_steps": 10,
    "save_steps": 100,
    "eval_steps": 100,
}

print(f"Model: {CONFIG['model_id']}")
print(f"LoRA rank: {CONFIG['lora_rank']}, alpha: {CONFIG['lora_alpha']}")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"Max sequence length: {CONFIG['max_seq_length']}")
print(f"Epochs: {CONFIG['num_epochs']} (≈188 steps, ≈2.5 hrs on T4)")
print(f"Mixed precision: fp16={CONFIG['fp16']}, bf16={CONFIG['bf16']} (both OFF — avoids GradScaler crash)")

## Step 3: Generate Synthetic Training Data

This cell contains the full data generator. It creates realistic Indian hospital documents
with various messiness levels (clean text, OCR noise, bad scans, phone photos).

- **1500 training** / **150 validation** / **400 test** cases
- Each case = 1 claim form + 1 discharge summary
- 30% of cases have deliberate mismatches (diagnosis swap, amount inflation, etc.)
- 5 noise levels simulating OCR quality


In [ ]:
# ── Step 3: Generate synthetic data ───────────────────────────────────────
# Full data generator (Indian hospital claim forms & discharge summaries)
# Output: data/synth/{train,val,test}.jsonl

"""
ClaimCheck Synthetic Data Generator
Generates realistic Indian hospital claim forms and discharge summaries
with various messiness levels (clean, OCR noise, bad scans, phone photos).

Each case = 1 claim form + 1 discharge summary.
Some cases have deliberate mismatches for the comparer to catch.

Output: data/synth/{train,val,test}.jsonl
Each line: {"case_id": ..., "doc_type": "claim_form"|"discharge_summary",
            "text": ..., "messiness": ..., "fields": {...}, "has_mismatch": bool}
"""

import json
import random
import os
import string
from pathlib import Path

random.seed(42)

# ── Indian-realistic reference data ──────────────────────────────────────

FIRST_NAMES_M = [
    "Rajesh", "Amit", "Suresh", "Vikram", "Arun", "Manoj", "Deepak", "Sanjay",
    "Ravi", "Prakash", "Kiran", "Mohan", "Rahul", "Nikhil", "Sachin", "Venkat",
    "Ashok", "Gopal", "Ramesh", "Naresh", "Harish", "Ganesh", "Mahesh", "Dinesh",
    "Ajay", "Vijay", "Sunil", "Anil", "Mukesh", "Rakesh", "Pavan", "Srikanth",
]
FIRST_NAMES_F = [
    "Priya", "Sunita", "Lakshmi", "Anita", "Kavitha", "Meena", "Radha", "Sita",
    "Geeta", "Pooja", "Neha", "Asha", "Jaya", "Rekha", "Swati", "Divya",
    "Nirmala", "Padma", "Sarala", "Usha", "Vidya", "Kamala", "Shanti", "Latha",
    "Deepa", "Savitha", "Bhavani", "Aruna", "Suma", "Revathi", "Vani", "Rani",
]
LAST_NAMES = [
    "Sharma", "Reddy", "Kumar", "Patel", "Singh", "Rao", "Gupta", "Verma",
    "Nair", "Pillai", "Iyer", "Das", "Mishra", "Joshi", "Bhat", "Choudhury",
    "Malhotra", "Mehta", "Banerjee", "Mukherjee", "Chatterjee", "Sen", "Dey",
    "Patil", "Deshmukh", "Kulkarni", "Jain", "Agarwal", "Tiwari", "Pandey",
    "Saxena", "Kapoor", "Menon", "Kaur", "Rajan", "Hegde", "Shetty", "Gowda",
]

HOSPITALS = [
    "Apollo Hospitals", "Fortis Hospital", "Max Super Speciality Hospital",
    "AIIMS", "Yashoda Hospital", "KIMS Hospital", "Care Hospital",
    "Manipal Hospital", "Narayana Health", "Medanta The Medicity",
    "Sir Ganga Ram Hospital", "Lilavati Hospital", "Global Hospital",
    "Continental Hospital", "Sunshine Hospital", "Star Hospital",
    "Aster CMI Hospital", "Columbia Asia Hospital", "Sakra World Hospital",
    "Sagar Hospital", "BGS Gleneagles Hospital", "Sparsh Hospital",
]

CITIES = [
    "Hyderabad", "Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata",
    "Pune", "Ahmedabad", "Jaipur", "Lucknow", "Coimbatore", "Vizag",
    "Kochi", "Nagpur", "Chandigarh", "Bhopal", "Indore", "Mysore",
]

DOCTOR_TITLES = ["Dr.", "Dr"]
DOCTOR_SPECIALITIES = [
    "General Medicine", "Cardiology", "Orthopedics", "Neurology",
    "Gastroenterology", "Pulmonology", "Nephrology", "Urology",
    "General Surgery", "ENT", "Ophthalmology", "Dermatology",
    "Oncology", "Endocrinology", "Rheumatology", "Psychiatry",
]

# Diagnosis-procedure pairs (matching and mismatching)
DIAGNOSIS_PROCEDURE_PAIRS = [
    ("Acute Myocardial Infarction", "Percutaneous Coronary Intervention (PCI)"),
    ("Acute Myocardial Infarction", "Coronary Artery Bypass Grafting (CABG)"),
    ("Cholelithiasis", "Laparoscopic Cholecystectomy"),
    ("Appendicitis", "Laparoscopic Appendectomy"),
    ("Lumbar Disc Herniation", "Lumbar Discectomy"),
    ("Fracture Neck of Femur", "Hip Replacement Surgery"),
    ("Cataract", "Phacoemulsification with IOL Implantation"),
    ("Benign Prostatic Hyperplasia", "Transurethral Resection of Prostate (TURP)"),
    ("Inguinal Hernia", "Laparoscopic Hernia Repair"),
    ("Uterine Fibroids", "Laparoscopic Myomectomy"),
    ("Chronic Kidney Disease Stage 5", "Arteriovenous Fistula Creation"),
    ("Diabetic Foot Ulcer", "Wound Debridement and Skin Grafting"),
    ("Osteoarthritis of Knee", "Total Knee Replacement"),
    ("Cervical Spondylosis", "Anterior Cervical Discectomy and Fusion"),
    ("Pleural Effusion", "Intercostal Drainage"),
    ("Renal Calculus", "Percutaneous Nephrolithotomy (PCNL)"),
    ("Varicose Veins", "Endovenous Laser Ablation"),
    ("Nasal Septal Deviation", "Septoplasty"),
    ("Chronic Tonsillitis", "Tonsillectomy"),
    ("Breast Carcinoma", "Modified Radical Mastectomy"),
    ("Type 2 Diabetes Mellitus", "Medical Management"),
    ("Hypertension", "Medical Management"),
    ("Pneumonia", "Medical Management"),
    ("Dengue Fever", "Supportive Care"),
    ("Acute Gastroenteritis", "Medical Management"),
    ("Urinary Tract Infection", "Medical Management"),
    ("Bronchial Asthma Exacerbation", "Nebulization and Medical Management"),
    ("COPD Exacerbation", "Medical Management with Oxygen Therapy"),
    ("Deep Vein Thrombosis", "Anticoagulation Therapy"),
    ("Cellulitis", "IV Antibiotics and Medical Management"),
]

SECONDARY_DIAGNOSES = [
    "Type 2 Diabetes Mellitus", "Essential Hypertension", "Hypothyroidism",
    "Iron Deficiency Anemia", "Dyslipidemia", "Obesity", "GERD",
    "Chronic Kidney Disease Stage 2", "Mild Asthma", "Vitamin D Deficiency",
    "Benign Prostatic Hyperplasia", "Osteoporosis", "Allergic Rhinitis",
    None, None, None, None, None,  # many cases have no secondary
]

INSURANCE_COMPANIES = [
    "Star Health", "ICICI Lombard", "HDFC ERGO", "Max Bupa", "New India Assurance",
    "Bajaj Allianz", "Tata AIG", "Religare Health", "Aditya Birla Health",
    "SBI Health", "Oriental Insurance", "United India Insurance", "Niva Bupa",
]

CONDITION_AT_DISCHARGE = [
    "Stable and afebrile", "Hemodynamically stable", "Improved and stable",
    "Satisfactory condition", "Good general condition", "Stable, ambulatory",
    "Afebrile, tolerating oral feeds well", "Conscious, oriented, stable vitals",
]

FOLLOW_UP = [
    "Review after 1 week with reports",
    "Follow up in OPD after 2 weeks",
    "Review after 10 days, suture removal",
    "Follow up with treating doctor after 1 month",
    "Review in 1 week, repeat blood work",
    "OPD follow up after 15 days with X-ray",
    "Monthly follow up for 3 months",
    "Review after 6 weeks with repeat echocardiogram",
]


def random_date(year_range=(2023, 2026)):
    y = random.randint(*year_range)
    m = random.randint(1, 12)
    d = random.randint(1, 28)
    fmt = random.choice([
        f"{d:02d}/{m:02d}/{y}",
        f"{d:02d}-{m:02d}-{y}",
        f"{d}/{m}/{y}",
        f"{y}-{m:02d}-{d:02d}",
    ])
    return fmt, (y, m, d)


def random_policy_number():
    prefix = random.choice(["HI", "POL", "SH", "IC", "HD", "NI", "BA", "TA"])
    year = random.randint(2020, 2026)
    num = random.randint(10000, 99999)
    sep = random.choice(["-", "/", ""])
    return f"{prefix}{sep}{year}{sep}{num}"


def random_reg_number():
    prefix = random.choice(["IP", "REG", "MRN", "UHID", "HN"])
    num = random.randint(100000, 999999)
    return f"{prefix}/{num}"


def random_amount_range(low, high):
    amt = random.randint(low, high)
    amt = round(amt / 100) * 100  # round to nearest 100
    return amt


def generate_case(case_id, introduce_mismatch=False):
    """Generate one case: a claim form and discharge summary pair."""

    # Patient info
    gender = random.choice(["M", "F"])
    if gender == "M":
        first = random.choice(FIRST_NAMES_M)
    else:
        first = random.choice(FIRST_NAMES_F)
    last = random.choice(LAST_NAMES)
    patient_name = f"{first} {last}"
    age = random.randint(18, 85)

    # Hospital & doctor
    hospital = random.choice(HOSPITALS)
    city = random.choice(CITIES)
    doc_first = random.choice(FIRST_NAMES_M + FIRST_NAMES_F)
    doc_last = random.choice(LAST_NAMES)
    doc_title = random.choice(DOCTOR_TITLES)
    speciality = random.choice(DOCTOR_SPECIALITIES)
    doctor_name = f"{doc_title} {doc_first} {doc_last}"

    # Dates
    admit_str, admit_tuple = random_date()
    los = random.randint(1, 14)  # length of stay
    disc_y, disc_m, disc_d = admit_tuple
    disc_d = min(disc_d + los, 28)
    disc_str = admit_str.replace(str(admit_tuple[2]).zfill(2), str(disc_d).zfill(2), 1)
    # simple replacement for the day part

    # Diagnosis and procedure
    dx_idx = random.randint(0, len(DIAGNOSIS_PROCEDURE_PAIRS) - 1)
    diagnosis, procedure = DIAGNOSIS_PROCEDURE_PAIRS[dx_idx]
    secondary_dx = random.choice(SECONDARY_DIAGNOSES)

    # Insurance
    policy_num = random_policy_number()
    reg_num = random_reg_number()
    insurance_co = random.choice(INSURANCE_COMPANIES)

    # Amounts
    room_charges = random_amount_range(5000, 50000)
    investigation = random_amount_range(2000, 30000)
    medicine = random_amount_range(3000, 40000)
    consultation = random_amount_range(1000, 10000)
    procedure_charge = random_amount_range(10000, 200000) if "Surgery" in procedure or "ectomy" in procedure or "Replacement" in procedure or "PCI" in procedure or "CABG" in procedure or "TURP" in procedure or "PCNL" in procedure else random_amount_range(0, 5000)
    ot_charges = procedure_charge // 3 if procedure_charge > 5000 else 0
    total = room_charges + investigation + medicine + consultation + procedure_charge + ot_charges

    # Discharge info
    condition_discharge = random.choice(CONDITION_AT_DISCHARGE)
    follow_up = random.choice(FOLLOW_UP)

    # Introduce mismatch if requested
    mismatch_type = None
    claim_diagnosis = diagnosis
    claim_procedure = procedure
    disc_diagnosis = diagnosis
    disc_procedure = procedure
    claim_name = patient_name
    disc_name = patient_name
    claim_admit = admit_str
    disc_admit = admit_str

    if introduce_mismatch:
        mismatch_type = random.choice([
            "diagnosis", "procedure", "name_typo", "date", "amount_inflated"
        ])
        if mismatch_type == "diagnosis":
            # Claim says different diagnosis
            alt_idx = random.randint(0, len(DIAGNOSIS_PROCEDURE_PAIRS) - 1)
            while alt_idx == dx_idx:
                alt_idx = random.randint(0, len(DIAGNOSIS_PROCEDURE_PAIRS) - 1)
            claim_diagnosis = DIAGNOSIS_PROCEDURE_PAIRS[alt_idx][0]
        elif mismatch_type == "procedure":
            alt_idx = random.randint(0, len(DIAGNOSIS_PROCEDURE_PAIRS) - 1)
            while alt_idx == dx_idx:
                alt_idx = random.randint(0, len(DIAGNOSIS_PROCEDURE_PAIRS) - 1)
            claim_procedure = DIAGNOSIS_PROCEDURE_PAIRS[alt_idx][1]
        elif mismatch_type == "name_typo":
            # Subtle name difference
            claim_name = patient_name.replace(first[1], first[1].upper() if first[1].islower() else first[1].lower(), 1)
        elif mismatch_type == "date":
            # Off by a day or two
            off = random.choice([1, 2, -1])
            new_d = min(max(admit_tuple[2] + off, 1), 28)
            claim_admit = admit_str.replace(
                str(admit_tuple[2]).zfill(2),
                str(new_d).zfill(2), 1
            )
        elif mismatch_type == "amount_inflated":
            # Claim total higher than sum of itemized
            total = int(total * random.uniform(1.15, 1.40))

    # Ground truth fields for the SLM to extract
    claim_fields = {
        "patient_name": claim_name,
        "age": str(age),
        "gender": gender,
        "policy_number": policy_num,
        "insurance_company": insurance_co,
        "hospital_name": f"{hospital}, {city}",
        "date_of_admission": claim_admit,
        "date_of_discharge": disc_str,
        "primary_diagnosis": claim_diagnosis,
        "procedure": claim_procedure,
        "treating_doctor": doctor_name,
        "room_charges": str(room_charges),
        "investigation_charges": str(investigation),
        "medicine_charges": str(medicine),
        "consultation_fees": str(consultation),
        "procedure_charges": str(procedure_charge),
        "ot_charges": str(ot_charges),
        "total_amount": str(total),
    }
    if secondary_dx:
        claim_fields["secondary_diagnosis"] = secondary_dx

    discharge_fields = {
        "patient_name": disc_name,
        "age": str(age),
        "gender": gender,
        "registration_number": reg_num,
        "hospital_name": f"{hospital}, {city}",
        "date_of_admission": disc_admit,
        "date_of_discharge": disc_str,
        "admitting_diagnosis": disc_diagnosis,
        "final_diagnosis": disc_diagnosis,
        "procedures_performed": disc_procedure,
        "treating_doctor": doctor_name,
        "speciality": speciality,
        "condition_at_discharge": condition_discharge,
        "follow_up": follow_up,
    }
    if secondary_dx:
        discharge_fields["secondary_diagnosis"] = secondary_dx

    # Generate document text
    claim_text = generate_claim_form_text(claim_fields)
    discharge_text = generate_discharge_summary_text(discharge_fields)

    # Build quotes dict (exact text spans from the generated document)
    claim_quotes = build_quotes(claim_text, claim_fields)
    discharge_quotes = build_quotes(discharge_text, discharge_fields)

    return {
        "case_id": case_id,
        "has_mismatch": introduce_mismatch,
        "mismatch_type": mismatch_type,
        "claim": {
            "doc_type": "claim_form",
            "text": claim_text,
            "fields": claim_fields,
            "quotes": claim_quotes,
        },
        "discharge": {
            "doc_type": "discharge_summary",
            "text": discharge_text,
            "fields": discharge_fields,
            "quotes": discharge_quotes,
        },
    }


def generate_claim_form_text(fields):
    """Generate realistic claim form text with format variations."""
    style = random.randint(0, 2)

    if style == 0:
        # Structured form style
        lines = [
            f"HEALTH INSURANCE CLAIM FORM",
            f"",
            f"Insurance Company: {fields['insurance_company']}",
            f"Policy Number: {fields['policy_number']}",
            f"",
            f"PATIENT DETAILS",
            f"Patient Name: {fields['patient_name']}",
            f"Age: {fields['age']} years    Gender: {fields['gender']}",
            f"",
            f"HOSPITAL DETAILS",
            f"Hospital Name: {fields['hospital_name']}",
            f"Treating Doctor: {fields['treating_doctor']}",
            f"",
            f"ADMISSION DETAILS",
            f"Date of Admission: {fields['date_of_admission']}",
            f"Date of Discharge: {fields['date_of_discharge']}",
            f"",
            f"DIAGNOSIS AND TREATMENT",
            f"Primary Diagnosis: {fields['primary_diagnosis']}",
        ]
        if "secondary_diagnosis" in fields:
            lines.append(f"Secondary Diagnosis: {fields['secondary_diagnosis']}")
        lines += [
            f"Procedure/Treatment: {fields['procedure']}",
            f"",
            f"BILLING DETAILS",
            f"Room Charges: Rs. {fields['room_charges']}",
            f"Investigation Charges: Rs. {fields['investigation_charges']}",
            f"Medicine Charges: Rs. {fields['medicine_charges']}",
            f"Consultation Fees: Rs. {fields['consultation_fees']}",
            f"Procedure Charges: Rs. {fields['procedure_charges']}",
            f"OT Charges: Rs. {fields['ot_charges']}",
            f"",
            f"TOTAL AMOUNT CLAIMED: Rs. {fields['total_amount']}",
            f"",
            f"Declaration: I hereby declare that the above information is true.",
            f"Signature: _______________",
        ]

    elif style == 1:
        # Slightly different labels
        lines = [
            f"CASHLESS / REIMBURSEMENT CLAIM FORM",
            f"Claim Reference: CLM-{random.randint(100000,999999)}",
            f"",
            f"Insurer: {fields['insurance_company']}",
            f"Policy No.: {fields['policy_number']}",
            f"",
            f"Name of Patient: {fields['patient_name']}",
            f"Age/Gender: {fields['age']}/{fields['gender']}",
            f"",
            f"Name of Hospital: {fields['hospital_name']}",
            f"Attending Physician: {fields['treating_doctor']}",
            f"",
            f"Admitted On: {fields['date_of_admission']}",
            f"Discharged On: {fields['date_of_discharge']}",
            f"",
            f"Diagnosis: {fields['primary_diagnosis']}",
        ]
        if "secondary_diagnosis" in fields:
            lines.append(f"Co-morbidity: {fields['secondary_diagnosis']}")
        lines += [
            f"Surgical Procedure / Treatment Given: {fields['procedure']}",
            f"",
            f"--- ITEMIZED BILL ---",
            f"Room & Boarding         : Rs. {fields['room_charges']}/-",
            f"Investigations/Lab      : Rs. {fields['investigation_charges']}/-",
            f"Medicines & Consumables : Rs. {fields['medicine_charges']}/-",
            f"Doctor Consultation     : Rs. {fields['consultation_fees']}/-",
            f"Procedure/Surgery       : Rs. {fields['procedure_charges']}/-",
            f"Operation Theatre       : Rs. {fields['ot_charges']}/-",
            f"---",
            f"Grand Total: Rs. {fields['total_amount']}/-",
        ]

    else:
        # Paragraph / letter style
        lines = [
            f"To,",
            f"The Claims Department",
            f"{fields['insurance_company']}",
            f"",
            f"Subject: Health Insurance Claim under Policy {fields['policy_number']}",
            f"",
            f"Dear Sir/Madam,",
            f"",
            f"I, {fields['patient_name']}, aged {fields['age']} years ({fields['gender']}), "
            f"was admitted to {fields['hospital_name']} on {fields['date_of_admission']} "
            f"and discharged on {fields['date_of_discharge']}.",
            f"",
            f"I was treated by {fields['treating_doctor']} for {fields['primary_diagnosis']}.",
        ]
        if "secondary_diagnosis" in fields:
            lines.append(f"I also have {fields['secondary_diagnosis']} as a co-existing condition.")
        lines += [
            f"The procedure performed was {fields['procedure']}.",
            f"",
            f"The total hospital bill amounts to Rs. {fields['total_amount']}, "
            f"broken down as follows:",
            f"  Room Charges: Rs. {fields['room_charges']}",
            f"  Investigation: Rs. {fields['investigation_charges']}",
            f"  Medicines: Rs. {fields['medicine_charges']}",
            f"  Consultation: Rs. {fields['consultation_fees']}",
            f"  Procedure: Rs. {fields['procedure_charges']}",
            f"  OT: Rs. {fields['ot_charges']}",
            f"",
            f"Kindly process the claim at the earliest.",
            f"",
            f"Thanking you,",
            f"{fields['patient_name']}",
        ]

    return "\n".join(lines)


def generate_discharge_summary_text(fields):
    """Generate realistic discharge summary text."""
    style = random.randint(0, 2)

    if style == 0:
        lines = [
            f"DISCHARGE SUMMARY",
            f"{fields['hospital_name']}",
            f"",
            f"Patient Name: {fields['patient_name']}",
            f"Age/Gender: {fields['age']} yrs / {fields['gender']}",
            f"Reg. No.: {fields['registration_number']}",
            f"",
            f"Date of Admission: {fields['date_of_admission']}",
            f"Date of Discharge: {fields['date_of_discharge']}",
            f"",
            f"Treating Doctor: {fields['treating_doctor']} ({fields['speciality']})",
            f"",
            f"Admitting Diagnosis: {fields['admitting_diagnosis']}",
            f"Final Diagnosis: {fields['final_diagnosis']}",
        ]
        if "secondary_diagnosis" in fields:
            lines.append(f"Secondary Diagnosis: {fields['secondary_diagnosis']}")
        lines += [
            f"",
            f"Procedures Performed: {fields['procedures_performed']}",
            f"",
            f"CLINICAL SUMMARY:",
            f"Patient presented with complaints related to {fields['final_diagnosis']}. "
            f"After thorough evaluation and investigations, "
            f"{fields['procedures_performed']} was performed. "
            f"Post-operative/treatment course was uneventful.",
            f"",
            f"Condition at Discharge: {fields['condition_at_discharge']}",
            f"",
            f"FOLLOW UP: {fields['follow_up']}",
            f"",
            f"Treating Doctor: {fields['treating_doctor']}",
            f"Signature & Seal",
        ]

    elif style == 1:
        lines = [
            f"--- DISCHARGE SUMMARY ---",
            f"Hospital: {fields['hospital_name']}",
            f"MRN: {fields['registration_number']}",
            f"",
            f"PATIENT INFORMATION",
            f"  Name       : {fields['patient_name']}",
            f"  Age        : {fields['age']}",
            f"  Sex        : {fields['gender']}",
            f"  DOA        : {fields['date_of_admission']}",
            f"  DOD        : {fields['date_of_discharge']}",
            f"  Consultant : {fields['treating_doctor']}, Dept. of {fields['speciality']}",
            f"",
            f"DIAGNOSIS",
            f"  Primary    : {fields['admitting_diagnosis']}",
            f"  Final      : {fields['final_diagnosis']}",
        ]
        if "secondary_diagnosis" in fields:
            lines.append(f"  Associated : {fields['secondary_diagnosis']}")
        lines += [
            f"",
            f"PROCEDURE",
            f"  {fields['procedures_performed']}",
            f"",
            f"COURSE IN HOSPITAL",
            f"  Patient was managed for {fields['final_diagnosis']}. "
            f"  {fields['procedures_performed']} was done. "
            f"  Recovery was satisfactory.",
            f"",
            f"CONDITION AT DISCHARGE: {fields['condition_at_discharge']}",
            f"",
            f"ADVICE ON DISCHARGE:",
            f"  - {fields['follow_up']}",
            f"  - Continue prescribed medications",
            f"  - Report to ER if any warning signs",
            f"",
            f"  {fields['treating_doctor']}",
            f"  Department of {fields['speciality']}",
        ]

    else:
        # Narrative paragraph style
        lines = [
            f"DISCHARGE SUMMARY",
            f"{fields['hospital_name']}",
            f"Registration No: {fields['registration_number']}",
            f"",
            f"{fields['patient_name']}, {fields['age']} year old {fields['gender']}, "
            f"was admitted on {fields['date_of_admission']} under the care of "
            f"{fields['treating_doctor']} ({fields['speciality']}) "
            f"with a diagnosis of {fields['admitting_diagnosis']}.",
        ]
        if "secondary_diagnosis" in fields:
            lines.append(f"Known case of {fields['secondary_diagnosis']}.")
        lines += [
            f"",
            f"During the hospital stay, {fields['procedures_performed']} was performed. "
            f"The patient tolerated the procedure well and had an uneventful recovery.",
            f"",
            f"Final Diagnosis: {fields['final_diagnosis']}",
            f"Procedure Done: {fields['procedures_performed']}",
            f"",
            f"The patient is being discharged on {fields['date_of_discharge']} "
            f"in {fields['condition_at_discharge'].lower()} condition.",
            f"",
            f"Advised: {fields['follow_up']}",
            f"",
            f"Prepared by: {fields['treating_doctor']}",
        ]

    return "\n".join(lines)


def build_quotes(text, fields):
    """Find exact quote spans for each field value in the document text."""
    quotes = {}
    for key, value in fields.items():
        # Search for the value in the text
        idx = text.find(value)
        if idx != -1:
            # Get the line containing the value for context
            line_start = text.rfind("\n", 0, idx)
            line_end = text.find("\n", idx)
            if line_start == -1:
                line_start = 0
            else:
                line_start += 1
            if line_end == -1:
                line_end = len(text)
            quote_line = text[line_start:line_end].strip()
            quotes[key] = quote_line
        else:
            # Value might be embedded in a sentence
            quotes[key] = value
    return quotes


# ── OCR noise simulation ────────────────────────────────────────────────

def add_ocr_noise(text, level="light"):
    """Add realistic OCR-style noise to text."""
    if level == "clean":
        return text

    chars = list(text)
    noise_rate = {
        "light": 0.005,       # text PDF with minor issues
        "moderate": 0.015,    # decent scan
        "heavy": 0.035,       # phone photo
        "terrible": 0.06,     # bad scan, crumpled paper
    }.get(level, 0.01)

    # Common OCR substitutions
    ocr_subs = {
        "l": ["1", "I", "|"],
        "1": ["l", "I", "|"],
        "0": ["O", "o"],
        "O": ["0", "Q"],
        "o": ["0", "a"],
        "5": ["S", "s"],
        "S": ["5", "$"],
        "8": ["B", "&"],
        "B": ["8", "R"],
        "rn": ["m"],
        "m": ["rn"],
        "I": ["l", "1", "|"],
        "e": ["c"],
        "c": ["e", "("],
        "a": ["o", "e"],
        ".": [",", ""],
        ",": [".", ""],
        ":": [";", "."],
        "/": ["1", "7", "|"],
    }

    result = []
    i = 0
    while i < len(chars):
        if random.random() < noise_rate:
            action = random.choices(
                ["substitute", "delete", "insert", "space"],
                weights=[0.5, 0.2, 0.15, 0.15]
            )[0]

            if action == "substitute":
                # Check for 2-char substitutions first
                bigram = text[i:i+2] if i + 1 < len(text) else ""
                if bigram in ocr_subs:
                    result.append(random.choice(ocr_subs[bigram]))
                    i += 2
                    continue
                elif chars[i] in ocr_subs:
                    result.append(random.choice(ocr_subs[chars[i]]))
                else:
                    result.append(chars[i])
            elif action == "delete":
                pass  # skip the character
            elif action == "insert":
                result.append(chars[i])
                result.append(random.choice([" ", ".", ",", chars[i]]))
            elif action == "space":
                if chars[i] == " ":
                    result.append("  ")  # double space
                else:
                    result.append(" " + chars[i])  # spurious space
            i += 1
        else:
            result.append(chars[i])
            i += 1

    noisy = "".join(result)

    # For heavy/terrible: add line-level corruption
    if level in ("heavy", "terrible"):
        lines = noisy.split("\n")
        corrupted = []
        for line in lines:
            if random.random() < 0.05:
                # garbled line
                line = line[:len(line)//2] + "".join(
                    random.choices(string.ascii_letters + " .", k=random.randint(3, 10))
                )
            if random.random() < 0.03:
                # duplicate fragment
                words = line.split()
                if len(words) > 3:
                    idx = random.randint(0, len(words) - 2)
                    words.insert(idx + 1, words[idx])
                    line = " ".join(words)
            corrupted.append(line)
        noisy = "\n".join(corrupted)

    return noisy


# ── Make long records ────────────────────────────────────────────────────

def pad_to_long_record(text, target_lines=80):
    """Add realistic filler to make a long record."""
    current_lines = text.count("\n") + 1
    if current_lines >= target_lines:
        return text

    filler_sections = [
        "\nINVESTIGATION RESULTS:\n"
        "  CBC: Hb {hb} g/dL, WBC {wbc}/cumm, Platelets {plt} lakhs/cumm\n"
        "  RBS: {rbs} mg/dL\n"
        "  Serum Creatinine: {cr} mg/dL\n"
        "  Blood Urea: {bu} mg/dL\n"
        "  Serum Electrolytes: Na+ {na} mEq/L, K+ {k} mEq/L\n"
        "  LFT: Bilirubin {bil} mg/dL, SGOT {sgot} U/L, SGPT {sgpt} U/L\n"
        "  PT/INR: {pt} sec / {inr}\n"
        "  Urine Routine: {urine}\n"
        "  Blood Group: {bg}\n"
        "  ECG: {ecg}\n"
        "  Chest X-Ray: {cxr}\n",

        "\nDAILY PROGRESS NOTES:\n"
        "  Day 1: Patient admitted, initial assessment done. Vitals stable.\n"
        "  Day 2: Investigations sent. Patient kept NBM. IV fluids started.\n"
        "  Day 3: Reports reviewed. Planned for procedure.\n"
        "  Day 4: Procedure performed. Post-op vitals stable.\n"
        "  Day 5: Tolerating oral feeds. Ambulation started.\n"
        "  Day 6: Wound healthy. Drain removed. Recovery satisfactory.\n"
        "  Day 7: Fit for discharge. Follow up explained.\n",

        "\nMEDICATIONS ON DISCHARGE:\n"
        "  1. Tab. Paracetamol 650mg - SOS for pain\n"
        "  2. Tab. Pantoprazole 40mg - once daily before breakfast\n"
        "  3. Tab. Cefixime 200mg - twice daily for 5 days\n"
        "  4. Tab. Metronidazole 400mg - thrice daily for 5 days\n"
        "  5. Cap. Omeprazole 20mg - once daily\n"
        "  6. Multivitamin tablets - once daily for 30 days\n",

        "\nVITAL SIGNS CHART:\n"
        "  Admission: BP {bp1}, Pulse {p1}/min, Temp {t1}F, SpO2 {spo1}%\n"
        "  Day 2: BP {bp2}, Pulse {p2}/min, Temp {t2}F, SpO2 {spo2}%\n"
        "  Day 3: BP {bp3}, Pulse {p3}/min, Temp {t3}F, SpO2 {spo3}%\n"
        "  Discharge: BP {bp4}, Pulse {p4}/min, Temp {t4}F, SpO2 {spo4}%\n",
    ]

    # Fill in random lab values
    for section in filler_sections:
        text += section.format(
            hb=round(random.uniform(9.0, 15.0), 1),
            wbc=random.randint(4000, 15000),
            plt=round(random.uniform(1.5, 4.0), 1),
            rbs=random.randint(80, 200),
            cr=round(random.uniform(0.7, 2.0), 1),
            bu=random.randint(15, 50),
            na=random.randint(135, 145),
            k=round(random.uniform(3.5, 5.0), 1),
            bil=round(random.uniform(0.3, 2.0), 1),
            sgot=random.randint(15, 80),
            sgpt=random.randint(15, 80),
            pt=round(random.uniform(11, 16), 1),
            inr=round(random.uniform(0.9, 1.4), 1),
            urine=random.choice(["NAD", "Trace albumin", "Few pus cells", "Normal"]),
            bg=random.choice(["A+", "B+", "O+", "AB+", "A-", "O-"]),
            ecg=random.choice(["Normal sinus rhythm", "Sinus tachycardia", "NSR, no ST changes"]),
            cxr=random.choice(["Normal", "Bilateral clear lung fields", "No significant abnormality"]),
            bp1=f"{random.randint(110,160)}/{random.randint(70,100)}",
            p1=random.randint(70, 110),
            t1=round(random.uniform(98.0, 101.0), 1),
            spo1=random.randint(94, 100),
            bp2=f"{random.randint(110,150)}/{random.randint(70,90)}",
            p2=random.randint(70, 100),
            t2=round(random.uniform(98.0, 100.0), 1),
            spo2=random.randint(95, 100),
            bp3=f"{random.randint(110,140)}/{random.randint(70,90)}",
            p3=random.randint(70, 95),
            t3=round(random.uniform(97.5, 99.5), 1),
            spo3=random.randint(96, 100),
            bp4=f"{random.randint(110,140)}/{random.randint(70,85)}",
            p4=random.randint(68, 90),
            t4=round(random.uniform(97.5, 99.0), 1),
            spo4=random.randint(97, 100),
        )
        if text.count("\n") + 1 >= target_lines:
            break

    return text


# ── Main generation ──────────────────────────────────────────────────────

def assign_messiness():
    """Random messiness level with realistic distribution."""
    return random.choices(
        ["clean", "light", "moderate", "heavy", "terrible"],
        weights=[0.25, 0.25, 0.20, 0.20, 0.10],
    )[0]


def generate_dataset(n_train=1500, n_val=150, n_test=400, output_dir="data/synth"):
    """Generate the full synthetic dataset."""
    os.makedirs(output_dir, exist_ok=True)

    splits = {
        "train": n_train,
        "val": n_val,
        "test": n_test,
    }

    case_id = 0
    for split_name, count in splits.items():
        filepath = os.path.join(output_dir, f"{split_name}.jsonl")
        mismatch_rate = 0.30  # 30% of cases have mismatches

        with open(filepath, "w") as f:
            for i in range(count):
                case_id += 1
                has_mismatch = random.random() < mismatch_rate
                case = generate_case(f"CASE-{case_id:05d}", introduce_mismatch=has_mismatch)

                # Assign messiness levels (independently per document)
                claim_mess = assign_messiness()
                disc_mess = assign_messiness()

                # Some cases are long records
                is_long = random.random() < 0.12
                claim_text = case["claim"]["text"]
                disc_text = case["discharge"]["text"]
                if is_long:
                    disc_text = pad_to_long_record(disc_text, target_lines=random.randint(60, 120))

                # Apply OCR noise
                claim_noisy = add_ocr_noise(claim_text, claim_mess)
                disc_noisy = add_ocr_noise(disc_text, disc_mess)

                # Write claim form example
                claim_example = {
                    "case_id": case["case_id"],
                    "doc_type": "claim_form",
                    "text": claim_noisy,
                    "clean_text": claim_text,
                    "messiness": claim_mess,
                    "fields": case["claim"]["fields"],
                    "quotes": case["claim"]["quotes"],
                    "has_mismatch": case["has_mismatch"],
                    "mismatch_type": case["mismatch_type"],
                }
                f.write(json.dumps(claim_example) + "\n")

                # Write discharge summary example
                disc_example = {
                    "case_id": case["case_id"],
                    "doc_type": "discharge_summary",
                    "text": disc_noisy,
                    "clean_text": disc_text,
                    "messiness": disc_mess,
                    "fields": case["discharge"]["fields"],
                    "quotes": case["discharge"]["quotes"],
                    "has_mismatch": case["has_mismatch"],
                    "mismatch_type": case["mismatch_type"],
                }
                f.write(json.dumps(disc_example) + "\n")

        doc_count = count * 2
        print(f"  {split_name}: {count} cases, {doc_count} documents -> {filepath}")

    print(f"\nTotal: {sum(splits.values())} cases, {sum(splits.values()) * 2} documents")
    print(f"Output: {output_dir}/")


# Alias for compatibility
generate_all = generate_dataset

# Run the generator
print("Generating ClaimCheck synthetic dataset...")
print("=" * 50)
generate_dataset()
print("\n✅ Synthetic data generated in data/synth/")

In [ ]:
# ── Step 4: Load data and format for Qwen2.5 chat template ───────────────
import json
from datasets import Dataset

SYSTEM_PROMPT = """You are ClaimCheck, a medical document field extractor for Indian hospital claims.

Given a hospital document (claim form or discharge summary), extract these fields with EXACT quotes from the text:
- patient_name: Full name of the patient
- age: Patient's age
- gender: Patient's gender
- hospital_name: Name of the hospital
- admission_date: Date of admission
- discharge_date: Date of discharge
- diagnosis: Primary/final diagnosis
- procedure: Main procedure/treatment performed
- total_amount: Total bill/claim amount (claim forms only; null for discharge summaries)

For EACH field, provide:
1. "value": The extracted value (cleaned up)
2. "quote": The EXACT text span from the document where you found this

Output valid JSON only. If a field is not found, set value to null and quote to null."""

# ── Field name mapping ───────────────────────────────────────────────────
# The data generator uses detailed field names per doc type.
# The model outputs canonical (unified) field names.
# This mapping translates: generator_field_name -> canonical_name

FIELD_MAP_CLAIM = {
    "patient_name":    "patient_name",
    "age":             "age",
    "gender":          "gender",
    "hospital_name":   "hospital_name",
    "date_of_admission": "admission_date",
    "date_of_discharge": "discharge_date",
    "primary_diagnosis": "diagnosis",
    "procedure":       "procedure",
    "total_amount":    "total_amount",
}

FIELD_MAP_DISCHARGE = {
    "patient_name":       "patient_name",
    "age":                "age",
    "gender":             "gender",
    "hospital_name":      "hospital_name",
    "date_of_admission":  "admission_date",
    "date_of_discharge":  "discharge_date",
    "final_diagnosis":    "diagnosis",
    "procedures_performed": "procedure",
    # discharge summaries have no total_amount
}

CANONICAL_FIELDS = [
    "patient_name", "age", "gender", "hospital_name",
    "admission_date", "discharge_date", "diagnosis",
    "procedure", "total_amount",
]

def format_target(fields, quotes, doc_type):
    """Build the JSON the model should output, mapping generator field names to canonical ones."""
    field_map = FIELD_MAP_CLAIM if doc_type == "claim_form" else FIELD_MAP_DISCHARGE

    # Build reverse lookup: canonical_name -> generator_field_name
    reverse = {v: k for k, v in field_map.items()}

    result = {}
    for canonical in CANONICAL_FIELDS:
        gen_key = reverse.get(canonical)
        if gen_key and gen_key in fields:
            result[canonical] = {
                "value": fields[gen_key],
                "quote": quotes.get(gen_key) if quotes else None,
            }
        else:
            result[canonical] = {
                "value": None,
                "quote": None,
            }
    return json.dumps(result, indent=2)

def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    items = []
    with open(path) as f:
        for line in f:
            items.append(json.loads(line))
    return items

def build_conversations(jsonl_path):
    """Convert JSONL data into Qwen2.5 ChatML conversations for SFT."""
    items = load_jsonl(jsonl_path)
    conversations = []
    for item in items:
        doc_type_label = item["doc_type"].replace("_", " ").title()
        user_msg = f"Extract all fields from this {doc_type_label}:\n\n{item['text']}"
        assistant_msg = format_target(item["fields"], item.get("quotes", {}), item["doc_type"])

        conv = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": assistant_msg}
            ]
        }
        conversations.append(conv)
    return conversations

# Build datasets
train_convs = build_conversations("data/synth/train.jsonl")
val_convs = build_conversations("data/synth/val.jsonl")
test_convs = build_conversations("data/synth/test.jsonl")

train_dataset = Dataset.from_list(train_convs)
val_dataset = Dataset.from_list(val_convs)
test_dataset = Dataset.from_list(test_convs)

print(f"Train: {len(train_dataset)} conversations")
print(f"Val:   {len(val_dataset)} conversations")
print(f"Test:  {len(test_dataset)} conversations")
print(f"\nSample conversation roles: {[m['role'] for m in train_convs[0]['messages']]}")
print(f"User message preview: {train_convs[0]['messages'][1]['content'][:200]}...")

# Verify field mapping works — check a claim and a discharge example
sample_claim = json.loads(train_convs[0]['messages'][2]['content'])
sample_disc = json.loads(train_convs[1]['messages'][2]['content'])
print(f"\n── Verification ──")
print(f"Claim form fields populated:      {sum(1 for v in sample_claim.values() if v['value'] is not None)}/{len(CANONICAL_FIELDS)}")
print(f"Discharge summary fields populated: {sum(1 for v in sample_disc.values() if v['value'] is not None)}/{len(CANONICAL_FIELDS)}")
print(f"Claim 'diagnosis':     {sample_claim['diagnosis']['value']}")
print(f"Claim 'admission_date': {sample_claim['admission_date']['value']}")
print(f"Discharge 'diagnosis':  {sample_disc['diagnosis']['value']}")
print(f"Discharge 'procedure':  {sample_disc['procedure']['value']}")
print(f"Discharge 'total_amount': {sample_disc['total_amount']['value']}  (expected: None)")


In [ ]:
# ── Step 5: Load model with 4-bit quantization ──────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {CONFIG['model_id']}...")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_id"],
    trust_remote_code=True,
    padding_side="right",
)

# Ensure pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",   # flash_attn not on T4
)
model.config.use_cache = False     # required for gradient checkpointing

# ── CRITICAL FIX: Override Qwen2.5's bfloat16 default ───────────────────
# Qwen2.5's config.json has torch_dtype=bfloat16. peft's
# prepare_model_for_kbit_training reads model.config.torch_dtype and
# casts non-quantized layers (embeddings, layernorms, lm_head) to THAT
# dtype. If it stays bfloat16, those layers produce bf16 gradients,
# which crash PyTorch's fp16 GradScaler on the T4 (no bf16 support).
# Force config to float16 so prepare_model_for_kbit_training does the
# right thing.
model.config.torch_dtype = torch.float16

# Print model size info
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded")
print(f"Total parameters: {total_params:,} ({total_params/1e9:.1f}B)")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Model config torch_dtype forced to: {model.config.torch_dtype}")

# Verify no bf16 params exist after loading
bf16_count = sum(1 for p in model.parameters() if p.dtype == torch.bfloat16)
print(f"Parameters in bfloat16: {bf16_count} (should be 0)")

In [ ]:
# ── Step 6: Setup LoRA adapters ──────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for QLoRA training
try:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
except TypeError:
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()

# ── Safety net: cast any bf16 params to fp16 AFTER prepare_model ─────────
# prepare_model_for_kbit_training may cast non-quantized layers to the
# model config's torch_dtype. We already forced config to float16, but
# as a belt-and-suspenders check, sweep for any stray bf16 params.
# This is the layer that was missing in previous attempts — we only cast
# trainable params before, but the ERROR comes from NON-trainable layers.
bf16_fixed = 0
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
        bf16_fixed += 1
if bf16_fixed > 0:
    print(f"⚠️  Cast {bf16_fixed} stray bfloat16 parameters to float16")
else:
    print(f"✅ No bfloat16 parameters found (good!)")

# LoRA config
lora_config = LoraConfig(
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["lora_target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Cast LoRA (trainable) params to float32 for stable gradient accumulation
# on the T4, which doesn't support bf16 grad scaling
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Final dtype audit — make sure nothing is bf16 anywhere
dtype_counts = {}
for name, param in model.named_parameters():
    dt = str(param.dtype)
    dtype_counts[dt] = dtype_counts.get(dt, 0) + 1

print(f"\n── Parameter dtype audit ──")
for dt, count in sorted(dtype_counts.items()):
    print(f"  {dt}: {count} params")

bf16_remaining = sum(1 for p in model.parameters() if p.dtype == torch.bfloat16)
assert bf16_remaining == 0, f"❌ Still have {bf16_remaining} bf16 params! GradScaler will crash."
print(f"✅ Zero bfloat16 parameters — safe for fp16 GradScaler on T4")

# Print trainable params
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")
print(f"All parameters:       {all_params:,} ({all_params/1e9:.1f}B)")
print(f"Trainable %:          {100 * trainable_params / all_params:.2f}%")
print(f"LoRA rank: {CONFIG['lora_rank']}, alpha: {CONFIG['lora_alpha']}")
print(f"Target modules: {CONFIG['lora_target_modules']}")

In [ ]:
# ── Step 7: Train with SFTTrainer ────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

# Calculate warmup steps manually (warmup_ratio can cause issues in some trl versions)
total_steps = (len(train_dataset) // (CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"])) * CONFIG["num_epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
print(f"Total training steps: ~{total_steps}, warmup steps: {warmup_steps}")

sft_config = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=warmup_steps,
    weight_decay=CONFIG["weight_decay"],
    lr_scheduler_type=CONFIG["lr_scheduler"],
    max_grad_norm=CONFIG["max_grad_norm"],
    # ── THE FIX: both fp16 and bf16 OFF ──────────────────────────────────
    # Qwen2.5's 4-bit quantized layers produce bfloat16 activations at
    # RUNTIME during the forward pass, regardless of stored param dtypes.
    # fp16=True enables PyTorch's GradScaler, which calls
    # _amp_foreach_non_finite_check_and_unscale_cuda — a CUDA kernel that
    # has NO bfloat16 implementation. So it crashes every time.
    # With both OFF, no GradScaler is created. The model is already 4-bit
    # so memory savings from mixed precision are minimal. Gradient
    # checkpointing keeps VRAM in check. Training is ~1.5x slower but
    # it WORKS.
    fp16=False,
    bf16=False,
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    eval_steps=CONFIG["eval_steps"],
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=CONFIG["max_seq_length"],   # trl v1.13+ uses max_length
    packing=False,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Steps per epoch: ~{len(train_dataset) // (CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps'])}")
print(f"Mixed precision: fp16=False, bf16=False (GradScaler disabled)")
print(f"\n🚀 Starting training...")

train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Total steps: {trainer.state.global_step}")

In [ ]:
# ── Step 8: Save LoRA adapter ────────────────────────────────────────────
import os

adapter_path = os.path.join(CONFIG["output_dir"], "final_adapter")

# Save just the LoRA weights (small! ~60MB)
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Check adapter size
total_size = 0
for f in os.listdir(adapter_path):
    fpath = os.path.join(adapter_path, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        total_size += size
        print(f"  {f}: {size/1e6:.1f} MB")

print(f"\n✅ Adapter saved to {adapter_path}")
print(f"Total adapter size: {total_size/1e6:.1f} MB")
print(f"(The full 3B model is NOT saved — only the small LoRA adapter weights)")

In [ ]:
# ── Step 9: Evaluate on test set ─────────────────────────────────────────
import json
import re

def extract_json_from_response(text):
    """Try to parse JSON from model output."""
    # Try direct parse
    try:
        return json.loads(text)
    except:
        pass
    # Try to find JSON block
    match = re.search(r'\{[\s\S]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    return None

def evaluate_extraction(model, tokenizer, test_data, max_samples=100):
    """Evaluate field extraction accuracy on test set."""
    model.eval()
    results = {
        "total": 0,
        "valid_json": 0,
        "field_correct": 0,
        "field_total": 0,
        "has_quotes": 0,
        "per_field": {},
    }

    # These are the CANONICAL field names — same as what format_target outputs
    fields_to_check = ["patient_name", "age", "gender", "hospital_name",
                        "diagnosis", "procedure", "admission_date",
                        "discharge_date", "total_amount"]

    for field in fields_to_check:
        results["per_field"][field] = {"correct": 0, "total": 0}

    samples = test_data[:max_samples]
    print(f"Evaluating on {len(samples)} test samples...")

    for i, sample in enumerate(samples):
        messages = sample["messages"][:2]  # system + user only

        # Format input
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True,
                          max_length=CONFIG["max_seq_length"]).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
        results["total"] += 1

        # Parse response
        predicted = extract_json_from_response(response)
        if predicted is None:
            if (i + 1) % 20 == 0:
                print(f"  [{i+1}/{len(samples)}] ...")
            continue

        results["valid_json"] += 1

        # Parse expected (from format_target output — already uses canonical names)
        expected = json.loads(sample["messages"][2]["content"])

        # Compare fields
        for field in fields_to_check:
            if field not in expected:
                continue
            results["per_field"][field]["total"] += 1
            results["field_total"] += 1

            if field in predicted:
                pred_val = predicted[field]
                exp_val = expected[field]

                # Handle nested {"value": ..., "quote": ...} format
                if isinstance(pred_val, dict):
                    pred_v = str(pred_val.get("value", "") or "").strip().lower()
                    if pred_val.get("quote"):
                        results["has_quotes"] += 1
                else:
                    pred_v = str(pred_val).strip().lower()

                if isinstance(exp_val, dict):
                    exp_v = str(exp_val.get("value", "") or "").strip().lower()
                else:
                    exp_v = str(exp_val).strip().lower()

                # Skip comparison if expected is null (e.g. total_amount on discharge)
                if exp_v in ("", "none", "null"):
                    if pred_v in ("", "none", "null"):
                        results["field_correct"] += 1
                        results["per_field"][field]["correct"] += 1
                    continue

                if pred_v and exp_v and (pred_v == exp_v or pred_v in exp_v or exp_v in pred_v):
                    results["field_correct"] += 1
                    results["per_field"][field]["correct"] += 1

        if (i + 1) % 20 == 0:
            print(f"  [{i+1}/{len(samples)}] valid_json={results['valid_json']}")

    return results

# Run evaluation
test_convs_for_eval = build_conversations("data/synth/test.jsonl")
eval_results = evaluate_extraction(model, tokenizer, test_convs_for_eval, max_samples=100)

print(f"\n{'='*50}")
print(f"EVALUATION RESULTS ({eval_results['total']} samples)")
print(f"{'='*50}")
print(f"Valid JSON output: {eval_results['valid_json']}/{eval_results['total']} "
      f"({100*eval_results['valid_json']/max(1,eval_results['total']):.1f}%)")
print(f"Overall field accuracy: {eval_results['field_correct']}/{eval_results['field_total']} "
      f"({100*eval_results['field_correct']/max(1,eval_results['field_total']):.1f}%)")
print(f"Responses with quotes: {eval_results['has_quotes']}")
print(f"\nPer-field accuracy:")
for field, counts in eval_results["per_field"].items():
    acc = 100 * counts["correct"] / max(1, counts["total"])
    print(f"  {field:20s}: {counts['correct']:3d}/{counts['total']:3d} ({acc:.1f}%)")


In [ ]:
# ── Step 10: Download adapter weights ────────────────────────────────────
# Run this cell to zip and download the LoRA adapter.

import shutil

adapter_path = os.path.join(CONFIG["output_dir"], "final_adapter")
zip_name = "claimcheck-qwen2.5-3b-lora-adapter"

# Create zip
shutil.make_archive(zip_name, 'zip', adapter_path)
print(f"✅ Created {zip_name}.zip")

# Download (works on both Colab and Kaggle)
try:
    from google.colab import files
    files.download(f"{zip_name}.zip")
    print("📥 Download started (Colab)")
except ImportError:
    # On Kaggle, the file will be in the output directory
    shutil.copy(f"{zip_name}.zip", "/kaggle/working/")
    print(f"📥 File saved to /kaggle/working/{zip_name}.zip")
    print("   Go to Output tab to download it.")

In [ ]:
# ── Step 11: Quick inference test (optional) ─────────────────────────────
# Test the fine-tuned model on a sample document.

test_doc = """
DISCHARGE SUMMARY
Patient Name: Rajesh Kumar Sharma
Age/Sex: 45/M
Hospital: Apollo Hospitals, Hyderabad
Date of Admission: 15-Mar-2024
Date of Discharge: 22-Mar-2024

Diagnosis: Acute Myocardial Infarction (STEMI)
Procedure: Percutaneous Coronary Intervention (PCI) with Drug-Eluting Stent

Clinical Summary:
Patient presented with severe chest pain and was diagnosed with STEMI.
Emergency PCI was performed successfully with placement of a drug-eluting stent
in the LAD artery.

Total Bill Amount: Rs. 3,85,000/-
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Extract all fields from this Discharge Summary:\n\n{test_doc}"}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt", truncation=True,
                  max_length=CONFIG["max_seq_length"]).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model output:")
print(response)

# Try to pretty-print
parsed = extract_json_from_response(response)
if parsed:
    print("\n✅ Valid JSON output!")
    print(json.dumps(parsed, indent=2))
else:
    print("\n⚠️ Could not parse JSON from output")


## Done! 🎉

**What you have now:**
- A QLoRA fine-tuned Qwen2.5-3B model (~12M trainable params, ~0.4% of 3B total)
- The LoRA adapter weights (small, ~50MB zip)
- Evaluation results on the test set

**To use the adapter later:**
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model = PeftModel.from_pretrained(base_model, "./claimcheck-qwen2.5-3b-lora-adapter")
```

**Next steps for ClaimCheck:**
1. Compare this model against other SLMs (Phi-3-mini, Qwen2.5-1.5B, etc.) for the assignment comparison
2. Integrate into the full ClaimCheck pipeline (OCR → extraction → comparison → verdict)
3. Test on real-world documents with actual OCR noise

**Assignment note:** The model IS 3B parameters. QLoRA only trains ~12M adapter weights
but the full model remains 3B params. During inference, the adapter merges into the base model.